---
id: "readme"
template: "markdown"
name: "readme"
position:
  x: 0
  y: -40
dimensions:
  width: 900
  height: 360
config:
  md: |-
    # CUSTOMER_PREP (converted from Dataiku DSS)
    Recipe `compute_customers_prepared` (Prepare, 10 processors).

    | DSS processor(s) | Lakeflow Designer |
    |---|---|
    | StringTransformer TRIM / TO_LOWER / TO_UPPER, FillEmptyWithValue, DateParser | `prep_clean` |
    | CreateColumnWithGREL full_name, customer_type | `prep_formulas` (after `prep_clean`: formulas see cleaned values) |
    | FilterOnValue status = A (KEEP_ROW) | `prep_filter_active` |
    | ColumnsSelector + ColumnRenamer | `prep_select` |

    DSS treats empty string and NULL both as *empty*: `FillEmptyWithValue` becomes `CASE WHEN x IS NULL OR x = '' ...`.
---

In [0]:
"""
id: "src_customers"
template: "source"
templateVersion: "1.0.0"
name: "src_customers"
position:
  x: 0
  y: 360
description:
  text: "Dataset customers (PostgreSQL erp.customers)."
  hash: ""
previewMode: "1000"
config:
  table_source:
    tableName: "daphnedemos_catalog.migration_skill_tests.src_customers"
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "daphnedemos_catalog.migration_skill_tests.src_customers"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["src_customers.data"] = out["data"]


In [0]:
"""
id: "prep_clean"
template: "transform"
templateVersion: "1.0.0"
name: "prep_clean"
position:
  x: 300
  y: 360
description:
  text: "Steps 1-5: TRIM, TO_LOWER(email), TO_UPPER(country, segment), FillEmpty(segment), DateParser [dd.MM.yyyy, yyyy-MM-dd]."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "customer_id"
    - "TRIM(first_name) AS first_name"
    - "TRIM(last_name) AS last_name"
    - "LOWER(TRIM(email)) AS email"
    - "UPPER(TRIM(country)) AS country"
    - "CASE WHEN segment IS NULL OR TRIM(segment) = '' THEN 'UNKNOWN' ELSE UPPER(TRIM(segment)) END AS segment"
    - "COALESCE(CAST(TRY_TO_TIMESTAMP(signup_date, 'dd.MM.yyyy') AS DATE), CAST(TRY_TO_TIMESTAMP(signup_date, 'yyyy-MM-dd') AS DATE)) AS signup_date_parsed"
    - "status"
input:
  - node: "src_customers"
    input_port: "data"
    output_port: "data"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "customer_id",
        "TRIM(first_name) AS first_name",
        "TRIM(last_name) AS last_name",
        "LOWER(TRIM(email)) AS email",
        "UPPER(TRIM(country)) AS country",
        "CASE WHEN segment IS NULL OR TRIM(segment) = '' THEN 'UNKNOWN' ELSE UPPER(TRIM(segment)) END AS segment",
        "COALESCE(CAST(TRY_TO_TIMESTAMP(signup_date, 'dd.MM.yyyy') AS DATE), CAST(TRY_TO_TIMESTAMP(signup_date, 'yyyy-MM-dd') AS DATE)) AS signup_date_parsed",
        "status"
    ]
}
inputs = {
    "data": ctx["src_customers.data"]
}
out = run(config, inputs, spark)
ctx["prep_clean.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "prep_formulas"
template: "transform"
templateVersion: "1.0.0"
name: "prep_formulas"
position:
  x: 600
  y: 360
description:
  text: "Steps 6-7: formulas full_name, customer_type."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "*"
    - "COALESCE(first_name, '') || ' ' || COALESCE(last_name, '') AS full_name"
    - "CASE WHEN segment = 'CORP' OR segment = 'SMB' THEN 'B2B' ELSE 'B2C' END AS customer_type"
input:
  - node: "prep_clean"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "*",
        "COALESCE(first_name, '') || ' ' || COALESCE(last_name, '') AS full_name",
        "CASE WHEN segment = 'CORP' OR segment = 'SMB' THEN 'B2B' ELSE 'B2C' END AS customer_type"
    ]
}
inputs = {
    "data": ctx["prep_clean.transformed_data"]
}
out = run(config, inputs, spark)
ctx["prep_formulas.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "prep_filter_active"
template: "filter"
templateVersion: "1.0.0"
name: "prep_filter_active"
position:
  x: 900
  y: 360
description:
  text: "Step 8: FilterOnValue status = A (KEEP_ROW, exact)."
  hash: ""
previewMode: "1000"
config:
  condition: "status = 'A'"
input:
  - node: "prep_formulas"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df}

    return {"filtered_data": df.filter(condition)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "status = 'A'"
}
inputs = {
    "data": ctx["prep_formulas.transformed_data"]
}
out = run(config, inputs, spark)
ctx["prep_filter_active.filtered_data"] = out["filtered_data"]


In [0]:
"""
id: "prep_select"
template: "transform"
templateVersion: "1.0.0"
name: "prep_select"
position:
  x: 1200
  y: 360
description:
  text: "Steps 9-10: ColumnsSelector (keep) + ColumnRenamer."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "customer_id"
    - "full_name"
    - "email"
    - "country"
    - "segment"
    - "customer_type"
    - "signup_date_parsed AS signup_date"
input:
  - node: "prep_filter_active"
    input_port: "data"
    output_port: "filtered_data"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "customer_id",
        "full_name",
        "email",
        "country",
        "segment",
        "customer_type",
        "signup_date_parsed AS signup_date"
    ]
}
inputs = {
    "data": ctx["prep_filter_active.filtered_data"]
}
out = run(config, inputs, spark)
ctx["prep_select.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "out_customers_prepared"
template: "output"
templateVersion: "1.0.0"
name: "out_customers_prepared"
position:
  x: 1500
  y: 360
description:
  text: "Managed dataset customers_prepared."
  hash: ""
previewMode: "full"
config:
  catalog: "daphnedemos_catalog"
  schema: "migration_skill_tests"
  table_name: "dss_01_customers_prepared"
input:
  - node: "prep_select"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "daphnedemos_catalog",
    "schema": "migration_skill_tests",
    "table_name": "dss_01_customers_prepared"
}
inputs = {
    "data": ctx["prep_select.transformed_data"]
}
out = run(config, inputs, spark)
